# SalamaRecover — Synthetic Data Generation

This notebook generates **realistic demo data** for SalamaRecover.

## Why You Need This

When you walk into Kenyatta National Hospital to pitch SalamaRecover,
you open the hospital dashboard. If it's empty — no patients, no data,
no charts — nobody takes you seriously.

This notebook generates:
1. **50 patient profiles** — Kenyan names, realistic demographics
2. **Check-in histories** — 7-14 days of daily check-ins per patient
3. **CSV files** — Ready to import into Supabase

## Important

This is **fake data** for demos and testing. It is NOT real patient data.
Every record is generated by Gemini based on clinical patterns.

## Cost: Ksh 0
Gemini free tier. Google Colab free tier.

---
## Step 0 — Setup

In [ ]:
!pip install -q google-generativeai pandas

In [ ]:
import google.generativeai as genai
import pandas as pd
import json
import random
import time
from datetime import datetime, timedelta

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('Loaded API key from Colab secrets')
except Exception:
    GEMINI_API_KEY = input('Enter your Gemini API key: ')

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(
    model_name='gemini-2.0-flash',
    generation_config={
        'temperature': 0.6,
        'max_output_tokens': 4096,
        'response_mime_type': 'application/json',
    },
)

print('Gemini ready for data generation')

---
## Step 1 — Generate Patient Profiles

We generate 50 realistic Kenyan patient profiles.

Gemini is told to use:
- Real Kenyan names (Kikuyu, Luo, Kalenjin, Luhya, Kamba, etc.)
- Realistic age distributions per surgery type
- Nairobi-area hospitals
- Common Kenyan food allergies

We generate in batches of 10 to stay within output limits.

In [ ]:
PATIENT_GENERATION_PROMPT = """Generate {count} realistic Kenyan patient profiles for
a post-surgical recovery app. These are demo records, not real patients.

REQUIREMENTS:
- Use authentic Kenyan names from diverse ethnic groups
  (Kikuyu, Luo, Kalenjin, Luhya, Kamba, Kisii, Meru, Somali, etc.)
- Age should match the surgery type:
  Caesarean: 18-42 (female only)
  Appendectomy: 15-55 (any gender)
  Hernia Repair: 25-70 (mostly male)
  Cholecystectomy: 30-65 (mostly female)
  Knee Replacement: 50-80 (any gender)
- Hospitals should be real Nairobi hospitals:
  Kenyatta National Hospital, Nairobi Hospital, Mbagathi Hospital,
  Aga Khan University Hospital, Mama Lucy Kibaki Hospital,
  Kenyatta University Teaching Hospital
- Blood types should follow East African distribution:
  O+ (most common ~47%), A+ (~27%), B+ (~20%), AB+ (~4%), negatives rare
- Allergies: most patients have none. A few have milk, eggs, or peanuts.
- Surgery dates should be within the last 1-30 days from today ({today})

Return a JSON array of objects, each with:
- name (string)
- age (int)
- gender ("Female" or "Male")
- surgery_type (string)
- surgery_date ("YYYY-MM-DD")
- hospital (string)
- surgeon (string — Kenyan doctor name)
- weight (float — kg, realistic for Kenyan adults)
- blood_type (string)
- allergies (array of strings, empty for most)
- phone (string — Kenyan format +254 7XX XXX XXX)

Respond with ONLY the JSON array, nothing else.
"""

print('Patient generation prompt ready')

In [ ]:
# Generate patients in batches of 10
all_patients = []
batches = 5  # 5 batches x 10 = 50 patients
today = datetime.now().strftime('%Y-%m-%d')

for batch in range(batches):
    print(f'Generating batch {batch + 1}/{batches}...')

    prompt = PATIENT_GENERATION_PROMPT.format(count=10, today=today)

    try:
        response = model.generate_content(prompt)
        patients = json.loads(response.text)

        # Add unique IDs
        for i, patient in enumerate(patients):
            patient['id'] = f'P{len(all_patients) + i + 1:03d}'

        all_patients.extend(patients)
        print(f'  Generated {len(patients)} patients (total: {len(all_patients)})')

    except Exception as e:
        print(f'  Error in batch {batch + 1}: {e}')

    time.sleep(2)  # Rate limit friendly

print(f'\nTotal patients generated: {len(all_patients)}')

In [ ]:
# Convert to DataFrame and inspect
df_patients = pd.DataFrame(all_patients)
print(f'Patient DataFrame: {df_patients.shape}')
print(f'\nColumns: {list(df_patients.columns)}')
print(f'\nSurgery type distribution:')
print(df_patients['surgery_type'].value_counts())
print(f'\nGender distribution:')
print(df_patients['gender'].value_counts())
print(f'\nHospital distribution:')
print(df_patients['hospital'].value_counts())
print(f'\nAge statistics:')
print(df_patients['age'].describe())
print(f'\nSample patients:')
df_patients[['id', 'name', 'age', 'gender', 'surgery_type', 'hospital']].head(10)

---
## Step 2 — Generate Check-In Histories

For each patient, we generate daily check-in records from their
surgery date until today. Each check-in has:
- pain_level (0-10)
- symptoms (list)
- mood (Good/Tired/Anxious/Low)
- risk_level (LOW/MEDIUM/HIGH/EMERGENCY)

Gemini generates these as realistic recovery trajectories:
- **Most patients (80%):** Pain starts high, decreases steadily. LOW risk.
- **Some patients (15%):** Pain decreases then plateaus or bumps up. MEDIUM risk.
- **Few patients (5%):** Develop complications — fever, wound issues. HIGH risk.

This creates a realistic distribution for the hospital dashboard.

In [ ]:
CHECKIN_GENERATION_PROMPT = """Generate a realistic post-surgical recovery check-in
history for this patient:

PATIENT:
- Name: {name}
- Age: {age}, Gender: {gender}
- Surgery: {surgery_type}
- Surgery date: {surgery_date}
- Recovery trajectory: {trajectory}

Generate check-in records for Day 1 through Day {num_days}.

CLINICAL REALISM RULES:
- Day 1-2: Pain is typically 5-8. Nausea and dizziness are normal (anaesthesia).
- Day 3-7: Pain should decrease by ~1 point every 1-2 days for normal recovery.
  This is the highest risk window for surgical site infection.
- Day 7-14: Pain should be 2-4 for normal recovery.
- Day 14+: Pain should be 0-2.

TRAJECTORY "{trajectory}":
- If "normal": Steady improvement. Pain decreases. No complications. Risk stays LOW.
  Mood goes from Tired → Good over time.
- If "slow_recovery": Pain decreases slower than normal. Some warning symptoms
  appear (mild swelling, constipation). Risk fluctuates LOW-MEDIUM. Mood: mostly Tired.
- If "complication": Recovery starts normal, then around Day 4-6 patient develops
  fever + wound redness (SSI). Pain increases. Risk goes to HIGH. Mood: Anxious.

SYMPTOMS must be chosen from this list ONLY:
Fever above 38°C, Wound bleeding, Nausea, Vomiting, Dizziness,
Mild headache, Swelling, Redness around wound, Wound discharge,
Constipation, Loss of appetite, Numbness or tingling

MOOD must be one of: Good, Tired, Anxious, Low

RISK LEVEL must be one of: LOW, MEDIUM, HIGH, EMERGENCY
- LOW: Normal recovery, pain <5, no warning symptoms
- MEDIUM: Slow recovery, pain 5-6, or minor warning symptoms
- HIGH: Possible complication, fever + wound signs, or pain 7+
- EMERGENCY: Multiple critical symptoms (rare — only for "complication" trajectory)

Return a JSON array of objects, each with:
- day (int)
- pain_level (int 0-10)
- symptoms (array of strings from the list above)
- mood (string)
- risk_level (string)

Respond with ONLY the JSON array.
"""

print('Check-in generation prompt ready')

In [ ]:
# Assign recovery trajectories
# 80% normal, 15% slow, 5% complication
trajectories = (
    ['normal'] * 40 +
    ['slow_recovery'] * 8 +
    ['complication'] * 2
)
random.shuffle(trajectories)

# Ensure we have enough for all patients
while len(trajectories) < len(all_patients):
    trajectories.append('normal')

# Show distribution
from collections import Counter
traj_counts = Counter(trajectories[:len(all_patients)])
print('Trajectory distribution:')
for t, c in traj_counts.items():
    print(f'  {t}: {c} patients ({c/len(all_patients)*100:.0f}%)')

In [ ]:
# Generate check-in histories for each patient
all_checkins = []

for i, patient in enumerate(all_patients):
    # Calculate days since surgery
    surgery_date = datetime.strptime(patient['surgery_date'], '%Y-%m-%d')
    today_dt = datetime.now()
    days_since = (today_dt - surgery_date).days
    num_days = max(min(days_since, 21), 3)  # Between 3 and 21 days

    trajectory = trajectories[i]

    print(f'Patient {i+1}/{len(all_patients)}: {patient["name"]} '
          f'({patient["surgery_type"]}, {num_days} days, {trajectory})')

    prompt = CHECKIN_GENERATION_PROMPT.format(
        name=patient['name'],
        age=patient['age'],
        gender=patient['gender'],
        surgery_type=patient['surgery_type'],
        surgery_date=patient['surgery_date'],
        trajectory=trajectory,
        num_days=num_days,
    )

    try:
        response = model.generate_content(prompt)
        checkins = json.loads(response.text)

        # Add patient_id and trajectory to each check-in
        for checkin in checkins:
            checkin['patient_id'] = patient['id']
            checkin['patient_name'] = patient['name']
            checkin['surgery_type'] = patient['surgery_type']
            checkin['trajectory'] = trajectory
            # Calculate actual date
            checkin_date = surgery_date + timedelta(days=checkin['day'])
            checkin['date'] = checkin_date.strftime('%Y-%m-%d')

        all_checkins.extend(checkins)
        print(f'  → {len(checkins)} check-ins generated')

    except Exception as e:
        print(f'  → ERROR: {e}')

    # Rate limiting — pause every 5 patients
    if (i + 1) % 5 == 0:
        time.sleep(3)

print(f'\nTotal check-ins generated: {len(all_checkins)}')

In [ ]:
# Convert to DataFrame and inspect
df_checkins = pd.DataFrame(all_checkins)
print(f'Check-in DataFrame: {df_checkins.shape}')
print(f'\nRisk level distribution:')
print(df_checkins['risk_level'].value_counts())
print(f'\nPain level statistics:')
print(df_checkins['pain_level'].describe())
print(f'\nMood distribution:')
print(df_checkins['mood'].value_counts())
print(f'\nTrajectory distribution:')
print(df_checkins.groupby('trajectory')['risk_level'].value_counts())

---
## Step 3 — Validate the Data

Before using this data, let's verify it makes clinical sense.
These checks catch obvious errors in the generated data.

In [ ]:
print('DATA VALIDATION')
print('=' * 60)
issues = 0

# Check 1: Pain should generally decrease over time for normal patients
normal_patients = df_checkins[df_checkins['trajectory'] == 'normal']
for pid in normal_patients['patient_id'].unique():
    patient_data = normal_patients[normal_patients['patient_id'] == pid].sort_values('day')
    if len(patient_data) >= 4:
        first_half = patient_data.head(len(patient_data)//2)['pain_level'].mean()
        second_half = patient_data.tail(len(patient_data)//2)['pain_level'].mean()
        if second_half > first_half + 1:
            print(f'  ⚠️ Patient {pid}: Pain INCREASING in normal trajectory '
                  f'({first_half:.1f} → {second_half:.1f})')
            issues += 1

print(f'\n  Check 1 (normal pain decreasing): '
      f'{"PASS" if issues == 0 else f"{issues} issues found"}')

# Check 2: Complication patients should have HIGH/EMERGENCY at some point
comp_patients = df_checkins[df_checkins['trajectory'] == 'complication']
comp_check = 0
for pid in comp_patients['patient_id'].unique():
    patient_data = comp_patients[comp_patients['patient_id'] == pid]
    has_high = any(patient_data['risk_level'].isin(['HIGH', 'EMERGENCY']))
    if not has_high:
        print(f'  ⚠️ Patient {pid}: Complication trajectory but no HIGH/EMERGENCY risk')
        comp_check += 1

print(f'  Check 2 (complications have HIGH risk): '
      f'{"PASS" if comp_check == 0 else f"{comp_check} issues"}')

# Check 3: Risk level should match symptoms
critical_symptoms = {'Fever above 38°C', 'Wound bleeding'}
risk_check = 0
for _, row in df_checkins.iterrows():
    symptoms = set(row.get('symptoms', []))
    has_critical = bool(symptoms & critical_symptoms)
    if has_critical and row['risk_level'] == 'LOW':
        risk_check += 1

print(f'  Check 3 (critical symptoms not marked LOW): '
      f'{"PASS" if risk_check == 0 else f"{risk_check} issues"}')

# Check 4: Caesarean patients should be female
gender_check = 0
for _, patient in df_patients.iterrows():
    if patient['surgery_type'] == 'Caesarean Section' and patient['gender'] != 'Female':
        print(f'  ⚠️ Patient {patient["id"]}: Caesarean but gender is {patient["gender"]}')
        gender_check += 1

print(f'  Check 4 (caesarean = female): '
      f'{"PASS" if gender_check == 0 else f"{gender_check} issues"}')

# Check 5: Pain should be 0-10
pain_range = df_checkins[
    (df_checkins['pain_level'] < 0) | (df_checkins['pain_level'] > 10)
]
print(f'  Check 5 (pain 0-10 range): '
      f'{"PASS" if len(pain_range) == 0 else f"{len(pain_range)} out of range"}')

total_issues = issues + comp_check + risk_check + gender_check + len(pain_range)
print(f'\n{"="*60}')
print(f'Total issues: {total_issues}')
if total_issues == 0:
    print('✅ All validation checks passed!')
else:
    print('⚠️ Some issues found — review above and re-generate if needed.')

---
## Step 4 — Visualise the Data

Quick charts to verify the data looks realistic.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SalamaRecover — Synthetic Data Overview', fontsize=14, fontweight='bold')

# Chart 1: Pain over time by trajectory
ax1 = axes[0, 0]
for traj in ['normal', 'slow_recovery', 'complication']:
    traj_data = df_checkins[df_checkins['trajectory'] == traj]
    avg_pain = traj_data.groupby('day')['pain_level'].mean()
    ax1.plot(avg_pain.index, avg_pain.values, label=traj, linewidth=2)
ax1.set_xlabel('Day since surgery')
ax1.set_ylabel('Average pain level')
ax1.set_title('Pain Trajectory by Recovery Type')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Chart 2: Risk level distribution
ax2 = axes[0, 1]
risk_counts = df_checkins['risk_level'].value_counts()
colors = {'LOW': '#00B37E', 'MEDIUM': '#FFB703', 'HIGH': '#E63946', 'EMERGENCY': '#B71C1C'}
bars = ax2.bar(risk_counts.index, risk_counts.values,
               color=[colors.get(r, '#999') for r in risk_counts.index])
ax2.set_title('Risk Level Distribution')
ax2.set_ylabel('Number of check-ins')

# Chart 3: Surgery type distribution
ax3 = axes[1, 0]
surgery_counts = df_patients['surgery_type'].value_counts()
ax3.barh(surgery_counts.index, surgery_counts.values, color='#0077B6')
ax3.set_title('Surgery Type Distribution')
ax3.set_xlabel('Number of patients')

# Chart 4: Mood distribution
ax4 = axes[1, 1]
mood_counts = df_checkins['mood'].value_counts()
mood_colors = {'Good': '#00B37E', 'Tired': '#FFB703', 'Anxious': '#E63946', 'Low': '#B71C1C'}
ax4.pie(mood_counts.values, labels=mood_counts.index, autopct='%1.0f%%',
        colors=[mood_colors.get(m, '#999') for m in mood_counts.index])
ax4.set_title('Mood Distribution')

plt.tight_layout()
plt.savefig('synthetic_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved to synthetic_data_overview.png')

---
## Step 5 — Spot Check Individual Patients

Pick a few patients and review their full recovery timeline.
This is where you verify the data tells a believable story.

In [ ]:
def show_patient_timeline(patient_id):
    """Display a single patient's full recovery story."""
    patient = df_patients[df_patients['id'] == patient_id].iloc[0]
    checkins = df_checkins[df_checkins['patient_id'] == patient_id].sort_values('day')

    print(f'\n{"="*60}')
    print(f'PATIENT: {patient["name"]} ({patient["id"]})')
    print(f'  Age: {patient["age"]}, Gender: {patient["gender"]}')
    print(f'  Surgery: {patient["surgery_type"]} on {patient["surgery_date"]}')
    print(f'  Hospital: {patient["hospital"]}')
    print(f'  Allergies: {patient["allergies"]}')
    print(f'  Trajectory: {checkins.iloc[0]["trajectory"]}')
    print(f'{"-"*60}')

    for _, row in checkins.iterrows():
        symptoms_str = ', '.join(row['symptoms']) if row['symptoms'] else 'None'
        risk_icon = {'LOW': '🟢', 'MEDIUM': '🟡', 'HIGH': '🔴', 'EMERGENCY': '🚨'}
        icon = risk_icon.get(row['risk_level'], '⚪')

        print(f'  Day {row["day"]:2d} │ Pain: {row["pain_level"]:2d}/10 │ '
              f'{icon} {row["risk_level"]:9s} │ Mood: {row["mood"]:8s} │ '
              f'{symptoms_str}')

    print(f'{"-"*60}')

In [ ]:
# Show a normal recovery patient
normal_ids = df_checkins[df_checkins['trajectory'] == 'normal']['patient_id'].unique()
if len(normal_ids) > 0:
    show_patient_timeline(normal_ids[0])

In [ ]:
# Show a slow recovery patient
slow_ids = df_checkins[df_checkins['trajectory'] == 'slow_recovery']['patient_id'].unique()
if len(slow_ids) > 0:
    show_patient_timeline(slow_ids[0])

In [ ]:
# Show a complication patient
comp_ids = df_checkins[df_checkins['trajectory'] == 'complication']['patient_id'].unique()
if len(comp_ids) > 0:
    show_patient_timeline(comp_ids[0])

---
## Step 6 — Save to CSV

Save the data as CSV files. These can be:
1. Imported into Supabase (via CSV import in the dashboard)
2. Used in future ML training notebooks
3. Shared with clinicians for review

In [ ]:
# Save patients CSV
df_patients.to_csv('salama_patients_demo.csv', index=False)
print(f'Saved: salama_patients_demo.csv ({len(df_patients)} patients)')

# Save check-ins CSV
df_checkins.to_csv('salama_checkins_demo.csv', index=False)
print(f'Saved: salama_checkins_demo.csv ({len(df_checkins)} check-ins)')

# Summary
print(f'\n{"="*60}')
print(f'SUMMARY')
print(f'  Patients: {len(df_patients)}')
print(f'  Check-ins: {len(df_checkins)}')
print(f'  Risk distribution:')
for risk, count in df_checkins['risk_level'].value_counts().items():
    pct = count / len(df_checkins) * 100
    print(f'    {risk}: {count} ({pct:.0f}%)')
print(f'\nFiles ready for Supabase import or ML training.')

In [ ]:
# Download files (in Google Colab)
try:
    from google.colab import files
    files.download('salama_patients_demo.csv')
    files.download('salama_checkins_demo.csv')
    print('Files downloaded!')
except:
    print('Not in Colab — files saved to current directory.')

print('\nNext notebook: 04_prompt_evaluation.ipynb')
print('  → Automated testing of AI response quality')